# SpaceX Falcon 9 Landing Prediction Project

This notebook covers **Section 1: Understand the Dataset** and **Section 2: Data Exploration (EDA)** of the SpaceX Falcon 9 first-stage landing success prediction project.

---

# 1. Understand the Dataset

The dataset is downloaded from Kaggle (`xjoannax88/spacex-falcon-9-launches`) and contains launch records for Falcon 9, including payload, launch sites, orbits, boosters, and landing outcomes.

### Target Variable
- **Booster landing**: This contains the outcome of the booster landing. We will derive our target variable **`Class`** from this, where `1` represents a successful landing (outcome containing the word "Success") and `0` represents an unsuccessful landing.

### Feature Classification
1. **Numerical Features**:
   - `Flight No.`: Sequential number of the launch.
   - `Payload mass`: Weight of the payload in kilograms.
2. **Categorical Features**:
   - `Launch site`: The specific launch pad location (e.g. Cape Canaveral, KSC LC-39A, VAFB SLC-4E).
   - `Payload`: Name/description of the spacecraft carried.
   - `Orbit`: Target orbit type (e.g. LEO, GTO, ISS, MEO).
   - `Customer`: Entity/organization booking the launch.
   - `Launch outcome`: Overall status of the launch launch phase (e.g. Success, Failure).
   - `Version Booster`: Falcon 9 booster version/model variant.
   - `Booster landing`: Specific landing outcome text description.
3. **Date/Time Features**:
   - `Date`: Date of the launch.
   - `Time`: UTC launch time.

## 1.1 Imports and Settings

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="darkgrid", palette="muted")

## 1.2 Load Dataset

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'spacex_web_scraped.csv')
df = pd.read_csv(DATA_PATH)
df.head()

# 2. Exploratory Data Analysis (EDA)

Perform initial exploration to understand the properties of the data, extract the target variable, and check for missing values and duplicates.

## 2.1 Dataset Shape & Information

In [ ]:
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nDataset Information:")
df.info()

## 2.2 Handle Missing Values & Duplicates

In [ ]:
print("Missing Values per Column:")
print(df.isnull().sum())

print(f"\nDuplicate Records: {df.duplicated().sum()}")

## 2.3 Derive Target Class (`Class`)
We extract whether the booster landing was successful or not from the `Booster landing` column.

In [ ]:
# Derive the Class target variable: 1 for landing success, 0 for failure
df['Class'] = df['Booster landing'].apply(lambda x: 1 if 'Success' in str(x) else 0)

print("Landing Class Counts:")
print(df['Class'].value_counts())

print("\nLanding Success Rate:")
print(f"{df['Class'].mean() * 100:.2f}%")

## 2.4 Class Distribution Visualization

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Class', hue='Class', legend=False)
plt.title('Landing Outcome Distribution (0 = Fail, 1 = Success)')
plt.xlabel('Landing Outcome')
plt.ylabel('Count')
plt.show()

## 2.5 Clean Numerical Columns
The `Payload mass` column might contain some non-numeric values (like 'U' for unknown). We need to convert it to numeric and handle missing values.

In [ ]:
# Clean Payload mass
df['Payload mass'] = pd.to_numeric(df['Payload mass'].astype(str).str.replace('~', '').str.replace(',', '').str.strip(), errors='coerce')
# Impute missing payload mass with median
df['Payload mass'] = df['Payload mass'].fillna(df['Payload mass'].median())

print("Payload Mass Summary Statistics:")
print(df['Payload mass'].describe())

## 2.6 Success Rates by Launch Site, Orbit, and Booster Version

In [ ]:
# 1. Success Rate by Launch Site
plt.figure(figsize=(10, 5))
sns.barplot(data=df, x='Launch site', y='Class', errorbar=None, hue='Launch site', legend=False)
plt.title('Landing Success Rate by Launch Site')
plt.ylabel('Success Rate')
plt.ylim(0, 1.05)
plt.show()

In [ ]:
# 2. Success Rate by Orbit Type
plt.figure(figsize=(12, 6))
sns.barplot(data=df, x='Orbit', y='Class', errorbar=None, hue='Orbit', legend=False)
plt.title('Landing Success Rate by Orbit Type')
plt.ylabel('Success Rate')
plt.ylim(0, 1.05)
plt.xticks(rotation=45)
plt.show()

In [ ]:
# 3. Success Rate by Booster Version
plt.figure(figsize=(14, 6))
# Look at top 10 booster versions
top_boosters = df['Version Booster'].value_counts().index[:10]
sns.barplot(data=df[df['Version Booster'].isin(top_boosters)], x='Version Booster', y='Class', errorbar=None, hue='Version Booster', legend=False)
plt.title('Landing Success Rate of Top 10 Booster Versions')
plt.ylabel('Success Rate')
plt.ylim(0, 1.05)
plt.xticks(rotation=45)
plt.show()

## 2.7 Success Rate Over Time (Launch Year)
We parse the `Date` column to extract the launch year and track the improvement in success rate.

In [ ]:
# Extract year from Date
# Dates in dataset might be formatted like '4 June 2010' or '8 December 2010'
df['Year'] = pd.to_datetime(df['Date'], errors='coerce').dt.year

# Plot success rate over years
yearly_stats = df.groupby('Year')['Class'].mean().reset_index()

plt.figure(figsize=(10, 5))
sns.lineplot(data=yearly_stats, x='Year', y='Class', marker='o', linewidth=2.5)
plt.title('SpaceX Landing Success Rate Trend (Over Years)')
plt.xlabel('Year')
plt.ylabel('Success Rate')
plt.ylim(-0.05, 1.05)
plt.show()

## 2.8 Payload Mass Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of payload mass
sns.histplot(data=df, x='Payload mass', kde=True, ax=axes[0])
axes[0].set_title('Distribution of Payload Mass')
axes[0].set_xlabel('Payload Mass (kg)')

# Boxplot of payload mass by Class
sns.boxplot(data=df, x='Class', y='Payload mass', hue='Class', legend=False, ax=axes[1])
axes[1].set_title('Payload Mass by Landing Outcome')
axes[1].set_xlabel('Landing Class')
axes[1].set_ylabel('Payload Mass (kg)')

plt.show()